In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Assuming CSRNet class is already defined from a previous cell or import.
# If not, you might need to redefine or import it properly.

# Re-define CSRNet here if it's not globally available from previous runs
class CSRNet(nn.Module):
    def __init__(self):
        super(CSRNet, self).__init__()

        self.frontend_feat = [64, 64, 'M', 128, 128, 'M',
                              256, 256, 256, 'M',
                              512, 512, 512]

        self.backend_feat = [512, 512, 512, 256, 128, 64]

        self.frontend = self.make_layers(self.frontend_feat)
        self.backend = self.make_layers(self.backend_feat, in_channels=512, dilation=True)

        self.output_layer = nn.Conv2d(64, 1, kernel_size=1)

        self._initialize_weights()

    def make_layers(self, cfg, in_channels=3, dilation=False):
        layers = []
        for v in cfg:
            if v == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                if dilation:
                    conv2d = nn.Conv2d(in_channels, v, kernel_size=3,
                                       padding=2, dilation=2)
                else:
                    conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
                layers += [conv2d, nn.ReLU(inplace=True)]
                in_channels = v
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        x = self.output_layer(x)
        return torch.relu(x)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CSRNet().to(device)

checkpoint_path = "/content/drive/MyDrive/CSRNet/checkpoints/best_csrnet.pth"

# Load the checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)

# Rename 'output' keys to 'output_layer' in the checkpoint
# This handles the mismatch between the loaded checkpoint and the current model definition
if 'output.weight' in checkpoint:
    checkpoint['output_layer.weight'] = checkpoint.pop('output.weight')
    print("Renamed 'output.weight' to 'output_layer.weight'")

if 'output.bias' in checkpoint:
    checkpoint['output_layer.bias'] = checkpoint.pop('output.bias')
    print("Renamed 'output.bias' to 'output_layer.bias'")

# Load the modified state dictionary into the model
model.load_state_dict(checkpoint)

print("✅ best_csrnet.pth loaded successfully with key renaming")


Renamed 'output.weight' to 'output_layer.weight'
Renamed 'output.bias' to 'output_layer.bias'
✅ best_csrnet.pth loaded successfully with key renaming


In [ ]:
for name, param in model.named_parameters():
    if "frontend" in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

print("Frontend frozen ✅")

Frontend frozen ✅


In [ ]:
trainable = sum(p.requires_grad for p in model.parameters())
total = sum(1 for _ in model.parameters())

print(f"Trainable layers: {trainable}/{total}")

Trainable layers: 14/34


In [ ]:
import torch.optim as optim

criterion = torch.nn.MSELoss().to(device)

print("Loss function: MSE")

Loss function: MSE


In [ ]:
learning_rate = 1e-5

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=learning_rate
)

print("Optimizer initialized with LR =", learning_rate)

Optimizer initialized with LR = 1e-05


In [ ]:
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5
)

print("Scheduler added ✅")

Scheduler added ✅


In [ ]:
!pip install h5py

In [ ]:
import os
import cv2
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
train_root = "/content/drive/MyDrive/CSRNet/datasets/ShanghaiTech/part_B/train_data"

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch.nn.functional as F
import numpy as np

class CrowdDataset(Dataset):
    def __init__(self, img_dir, den_dir):
        self.imgs = []
        for f_name in os.listdir(img_dir):
            if f_name.endswith(".jpg"):
                den_path = f"{den_dir}/{f_name.replace('.jpg','.h5')}"
                if os.path.exists(den_path):
                    self.imgs.append(f_name)

        self.img_dir = img_dir
        self.den_dir = den_dir

        self.transform = transforms.Compose([
            transforms.Resize((288,512)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],
                                 [0.229,0.224,0.225])
        ])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        name = self.imgs[idx]

        img = self.transform(
            Image.open(f"{self.img_dir}/{name}").convert("RGB")
        )

        den_path = f"{self.den_dir}/{name.replace('.jpg','.h5')}"
        try:
            with h5py.File(den_path, "r") as f:
                density = torch.from_numpy(
                    np.array(f["density"])
                ).unsqueeze(0).unsqueeze(0)
        except KeyError as e:
            print(f"Skipping: KeyError: {e} in file: {den_path}")
            return None, None # Return None for both image and density

        # Resize density to match CSRNet output (36x64)
        density = F.interpolate(
            density,
            size=(36,64),
            mode="bilinear",
            align_corners=False
        )

        # Preserve count
        density = density * 64

        density = density.squeeze(0)

        return img, density

def custom_collate_fn(batch):
    # Filter out None samples (those that caused an error in __getitem__)
    batch = list(filter(lambda x: x[0] is not None, batch))
    if len(batch) == 0:
        # If the entire batch was filtered out, return None for images and density_maps
        return None, None

    images, density_maps = zip(*batch)
    images = torch.stack(images, 0)
    density_maps = torch.stack(density_maps, 0)
    return images, density_maps

In [ ]:
train_img = "/content/drive/MyDrive/CSRNet/datasets/ShanghaiTech/part_B/train_data/images"
train_den = "/content/drive/MyDrive/CSRNet/datasets/ShanghaiTech/part_B/train_data/density_maps"

train_loader = DataLoader(
    CrowdDataset(train_img, train_den),
    batch_size=1,
    shuffle=True,
    collate_fn=custom_collate_fn
)

print("Dataset size:", len(train_loader))

Dataset size: 400


In [ ]:
num_epochs = 5   # fine-tuning only, not full training
best_loss = float("inf")

In [ ]:
images, density_maps = next(iter(train_loader))
print(images.shape)
print(density_maps.shape)
print(density_maps.sum())

torch.Size([1, 3, 288, 512])
torch.Size([1, 1, 36, 64])
tensor(56.4376)


In [ ]:
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for images, density_maps in train_loader:
        images = images.to(device)
        density_maps = density_maps.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, density_maps)
        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.6f}")

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/CSRNet/checkpoints/best_csrnet_finetuned.pth"
        )
        print("✅ Best fine-tuned model saved")

    scheduler.step()

Epoch [1/5] - Loss: 0.000684
✅ Best fine-tuned model saved
Epoch [2/5] - Loss: 0.000411
✅ Best fine-tuned model saved
Epoch [3/5] - Loss: 0.000313
✅ Best fine-tuned model saved
Epoch [4/5] - Loss: 0.000270
✅ Best fine-tuned model saved
Epoch [5/5] - Loss: 0.000225
✅ Best fine-tuned model saved


In [ ]:
# Load original model
original_model = CSRNet().to(device)
# Load checkpoint and apply renaming if necessary
original_checkpoint = torch.load("/content/drive/MyDrive/CSRNet/checkpoints/best_csrnet.pth", map_location=device)
if 'output.weight' in original_checkpoint:
    original_checkpoint['output_layer.weight'] = original_checkpoint.pop('output.weight')
if 'output.bias' in original_checkpoint:
    original_checkpoint['output_layer.bias'] = original_checkpoint.pop('output.bias')
original_model.load_state_dict(original_checkpoint)
original_model.eval()

# Load fine-tuned model
finetuned_model = CSRNet().to(device)
# The fine-tuned model was saved after the keys were corrected, so it should already have 'output_layer' keys.
# We still use map_location for consistency.
finetuned_model.load_state_dict(
    torch.load("/content/drive/MyDrive/CSRNet/checkpoints/best_csrnet_finetuned.pth", map_location=device)
)
finetuned_model.eval()

print("Both models loaded ✅")

Both models loaded ✅


In [ ]:
test_img = "/content/drive/MyDrive/CSRNet/datasets/ShanghaiTech/part_B/test_data/images"
test_den = "/content/drive/MyDrive/CSRNet/datasets/ShanghaiTech/part_B/test_data/density_maps"

test_loader = DataLoader(
    CrowdDataset(test_img, test_den),
    batch_size=1,
    shuffle=False,
    collate_fn=custom_collate_fn
)

print("Test samples:", len(test_loader))

Test samples: 316


In [ ]:
import math

def evaluate(model, loader):
    model.eval()
    mae = 0.0
    mse = 0.0
    num_samples_processed = 0

    with torch.no_grad():
        for images, density_maps in loader:
            if images is None and density_maps is None: # Skip batches that custom_collate_fn filtered out entirely
                continue

            images = images.to(device)
            density_maps = density_maps.to(device)

            outputs = model(images)

            pred_count = outputs.sum().item()
            gt_count = density_maps.sum().item()

            mae += abs(pred_count - gt_count)
            mse += (pred_count - gt_count) ** 2
            num_samples_processed += 1

    if num_samples_processed == 0:
        return 0.0, 0.0 # Avoid division by zero if all samples were skipped

    mae /= num_samples_processed
    mse = math.sqrt(mse / num_samples_processed)

    return mae, mse

In [ ]:
orig_mae, orig_mse = evaluate(original_model, test_loader)
ft_mae, ft_mse = evaluate(finetuned_model, test_loader)

print("Original Model:")
print("MAE:", orig_mae)
print("MSE:", orig_mse)

print("\nFine-Tuned Model:")
print("MAE:", ft_mae)
print("MSE:", ft_mse)

Skipping: KeyError: "Unable to synchronously open object (object 'density' doesn't exist)" in file: /content/drive/MyDrive/CSRNet/datasets/ShanghaiTech/part_B/test_data/density_maps/IMG_105.h5
Skipping: KeyError: "Unable to synchronously open object (object 'density' doesn't exist)" in file: /content/drive/MyDrive/CSRNet/datasets/ShanghaiTech/part_B/test_data/density_maps/IMG_105.h5
Original Model:
MAE: 22.205895078844495
MSE: 28.319264318071976

Fine-Tuned Model:
MAE: 9.593915256999788
MSE: 12.191602081826998
